In [2]:
import kagglehub
import os
import torch
import pandas as pd
import warnings
import numpy as np
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

/home/aitech/anaconda3/envs/BERT/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
!python -c "import torch; print('CUDA available' if torch.cuda.is_available() else 'CUDA not available')"

CUDA available


### Kinopoisk's movies reviews

In [6]:
checkpoint = "DeepPavlov/rubert-base-cased"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [50]:
# def tokenize_function(examples):
#     return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

In [9]:
dataset_path = kagglehub.dataset_download("mikhailklemin/kinopoisks-movies-reviews")
dataset_path = os.path.join(dataset_path, "dataset")

In [10]:
neg_dataset = load_dataset("text", data_dir=os.path.join(dataset_path , "neg"), split="train")
pos_dataset = load_dataset("text", data_dir=os.path.join(dataset_path, "pos"), split="train")
neu_dataset = load_dataset("text", data_dir=os.path.join(dataset_path, "neu"), split="train")

neg_dataset = neg_dataset.map(lambda x: {"label": 0})
pos_dataset = pos_dataset.map(lambda x: {"label": 1})
neu_dataset = neu_dataset.map(lambda x: {"label": 2})

In [11]:
kinopoisk_data = concatenate_datasets([neg_dataset, pos_dataset, neu_dataset])

kinopoisk_data = kinopoisk_data.shuffle(seed=42)
train_test_split = kinopoisk_data.train_test_split(test_size=0.2, seed=42)

kinopoisk_data = DatasetDict({
    "train": train_test_split["train"],
    "test": train_test_split["test"]
})

### Common

In [12]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

### DeepPavlov/rubert-base-cased

In [13]:
CHECKPOINT_1 = "DeepPavlov/rubert-base-cased"

In [14]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_1) # не забыть переопределить для каждой модели
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_1, num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments("test-trainer")

In [ ]:
tokenized_kinopoisk_data  = kinopoisk_data.map(tokenize_function, batched=True)
tokenized_kinopoisk_data !

Map: 100%|██████████| 336621/336621 [00:47<00:00, 7022.93 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1346480
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 336621
    })
})

In [79]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_kinopoisk_data["train"],
    eval_dataset=tokenized_kinopoisk_data["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [81]:
trainer.train()

KeyboardInterrupt: 

In [ ]:
predictions = trainer.predict(tokenized_kinopoisk_data["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
predictions

In [ ]:
preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
preds

In [ ]:
# test_labels = tokenized_kinopoisk_data['label']

In [ ]:
# accuracy = accuracy_score(y_true, y_pred)
# print(f'Accuracy: {accuracy:.3f}')

In [ ]:
accuracy = accuracy_score(tokenized_kinopoisk_data['test']['label'], preds)
print(f'Accuracy: {accuracy:.3f}')

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average=None)
print("Precision per class:", precision)
print("Recall per class:", recall)
print("F1-score per class:", f1)

In [ ]:
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='macro')
print(f"Macro-average Precision: {precision_macro:.4f}")
print(f"Macro-average Recall: {recall_macro:.4f}")
print(f"Macro-average F1-score: {f1_macro:.4f}")

In [ ]:
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='weighted')
print(f"Weighted-average Precision: {precision_weighted:.4f}")
print(f"Weighted-average Recall: {recall_weighted:.4f}")
print(f"Weighted-average F1-score: {f1_weighted:.4f}")

In [ ]:
cm = confusion_matrix(tokenized_kinopoisk_data['test']['label'], preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
trainer.save_model('./deeppavlov-bert')

### cointegrated/rubert-tiny

In [ ]:
CHECKPOINT_2 = "cointegrated/rubert-tiny"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_2)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_2, num_labels=3)

In [ ]:
tokenized_kinopoisk_data  = kinopoisk_data.map(tokenize_function, batched=True)
tokenized_kinopoisk_data 

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments("test-trainer")

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_kinopoisk_data["train"],
    eval_dataset=tokenized_kinopoisk_data["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_kinopoisk_data["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
accuracy = accuracy_score(tokenized_kinopoisk_data['test']['label'], preds)
print(f'Accuracy: {accuracy:.3f}')

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average=None)
print("Precision per class:", precision)
print("Recall per class:", recall)
print("F1-score per class:", f1)

In [ ]:
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='macro')
print(f"Macro-average Precision: {precision_macro:.4f}")
print(f"Macro-average Recall: {recall_macro:.4f}")
print(f"Macro-average F1-score: {f1_macro:.4f}")

In [ ]:
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='weighted')
print(f"Weighted-average Precision: {precision_weighted:.4f}")
print(f"Weighted-average Recall: {recall_weighted:.4f}")
print(f"Weighted-average F1-score: {f1_weighted:.4f}")

In [ ]:
cm = confusion_matrix(tokenized_kinopoisk_data['test']['label'], preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
trainer.save_model('./cointegrated-rubert-tiny')

### ai-forever/ruBert-base Сбер

In [ ]:
CHECKPOINT_3 = "ai-forever/ruBert-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_3)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_3, num_labels=3)

In [ ]:
tokenized_kinopoisk_data  = kinopoisk_data.map(tokenize_function, batched=True)
tokenized_kinopoisk_data 

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments("test-trainer")

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_kinopoisk_data["train"],
    eval_dataset=tokenized_kinopoisk_data["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_kinopoisk_data["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
tokenized_kinopoisk_data['test']['label']

In [ ]:
accuracy = accuracy_score(tokenized_kinopoisk_data['test']['label'], preds)
print(f'Accuracy: {accuracy:.3f}')

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average=None)
print("Precision per class:", precision)
print("Recall per class:", recall)
print("F1-score per class:", f1)

In [ ]:
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='macro')
print(f"Macro-average Precision: {precision_macro:.4f}")
print(f"Macro-average Recall: {recall_macro:.4f}")
print(f"Macro-average F1-score: {f1_macro:.4f}")

NameError: name 'tokenized_kinopoisk_data' is not defined

In [ ]:
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='weighted')
print(f"Weighted-average Precision: {precision_weighted:.4f}")
print(f"Weighted-average Recall: {recall_weighted:.4f}")
print(f"Weighted-average F1-score: {f1_weighted:.4f}")

NameError: name 'tokenized_kinopoisk_data' is not defined

In [ ]:
cm = confusion_matrix(tokenized_kinopoisk_data['test']['label'], preds)
print("Confusion Matrix:")
print(cm)

NameError: name 'tokenized_kinopoisk_data' is not defined

In [ ]:
trainer.save_model('./ai-forever-ruBert-base')

NameError: name 'trainer' is not defined

### ai-forever/ruBert-large Сбер

In [ ]:
CHECKPOINT_4 = "ai-forever/ruBert-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_4)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_3, num_labels=4)

In [ ]:
tokenized_kinopoisk_data  = kinopoisk_data.map(tokenize_function, batched=True)
tokenized_kinopoisk_data 

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments("test-trainer")

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_kinopoisk_data["train"],
    eval_dataset=tokenized_kinopoisk_data["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_kinopoisk_data["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
preds = np.argmax(predictions.predictions, axis=-1)

In [ ]:
accuracy = accuracy_score(tokenized_kinopoisk_data['test']['label'], preds)
print(f'Accuracy: {accuracy:.3f}')

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average=None)
print("Precision per class:", precision)
print("Recall per class:", recall)
print("F1-score per class:", f1)

In [ ]:
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='macro')
print(f"Macro-average Precision: {precision_macro:.4f}")
print(f"Macro-average Recall: {recall_macro:.4f}")
print(f"Macro-average F1-score: {f1_macro:.4f}")

In [ ]:
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(tokenized_kinopoisk_data['test']['label'], preds, average='weighted')
print(f"Weighted-average Precision: {precision_weighted:.4f}")
print(f"Weighted-average Recall: {recall_weighted:.4f}")
print(f"Weighted-average F1-score: {f1_weighted:.4f}")

In [ ]:
cm = confusion_matrix(tokenized_kinopoisk_data['test']['label'], preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
trainer.save_model('./ai-forever-ruBert-large')